## Non-Repeatable Data Preprocessing

* Run data preprocessing here so that the RAG pipelines can use directly without executing each time

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
from llama_index.core import Document
import psycopg2
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [2]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    doc = Document(
        text=context,
        metadata={
            "doc_name": idx
        }
    )
    documents.append(doc)

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


In [3]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

In [3]:
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")

conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("Vector extension enabled.")

cur.execute("DROP TABLE IF EXISTS rag_dr_embeddings;")
cur.execute("DROP TABLE IF EXISTS data_rag_dr_embeddings_llamindex;")
cur.execute("DROP TABLE IF EXISTS data_data_embeddings_bge_small_en_v1dot5;")

for i in range(len(all_config['embedding_model_settings'])):
    table_name = f"data_embeddings_{all_config['embedding_model_settings'][i]['name']\
                               .split('/')[-1].replace('-', '_').replace('.', 'dot')}"
    cur.execute(f"DROP TABLE IF EXISTS {table_name};")

cur.close()
conn.close()

Connected to database: railway
Vector extension enabled.


### Raw Data Storage

* So that during the deployment time, no need to install `datasets`, and raw data loading from Railway DB can be faster than loading from Huggingface

In [4]:
import os
import json
import psycopg2
import warnings
warnings.filterwarnings('ignore')


DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")

conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("""
    DROP TABLE IF EXISTS raw_datasets;
    CREATE TABLE raw_datasets (
    id SERIAL PRIMARY KEY,
    dataset_name TEXT,
    record_index INT,
    data JSONB,
    created_at TIMESTAMP DEFAULT NOW()
);
CREATE INDEX idx_dataset
ON raw_datasets (dataset_name, record_index);
"""
)

for idx, record in enumerate(fiqa_eval):
    query = """
    INSERT INTO raw_datasets (dataset_name, record_index, data)
    VALUES (%s, %s, %s)
    """
    cur.execute(query, (
        "FIQA Data",
        idx,
        json.dumps(record)
    ))

cur.close()
conn.close()

Connected to database: railway


In [6]:
# load the dataset to see
dataset_name = "FIQA Data"

conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("""
    SELECT data
    FROM raw_datasets
    WHERE dataset_name = %s
    ORDER BY record_index
""", (dataset_name,))
rows = cur.fetchall()

cur.close()
conn.close()

Connected to database: railway


In [7]:
print(len(rows))
print(rows[0])

30
({'answer': '\nThe best way to deposit a cheque issued to an associate in your business into your business account is to open a business account with the bank. You will need a state-issued "dba" certificate from the county clerk\'s office as well as an Employer ID Number (EIN) issued by the IRS. Once you have opened the business account, you can have the associate sign the back of the cheque and deposit it into the business account.', 'contexts': ['Just have the associate sign the back and then deposit it.  It\'s called a third party cheque and is perfectly legal.  I wouldn\'t be surprised if it has a longer hold period and, as always, you don\'t get the money if the cheque doesn\'t clear. Now, you may have problems if it\'s a large amount or you\'re not very well known at the bank.  In that case you can have the associate go to the bank and endorse it in front of the teller with some ID.  You don\'t even technically have to be there.  Anybody can deposit money to your account if th

### Create Hash Mapping Table

* Save dataset & RAG settings & RAG output
* When using `UNIQUE`, psql automatically creates index on it, no need to create the index for the table 🍀

In [ ]:
import os
import psycopg2
import warnings
warnings.filterwarnings('ignore')


DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")

conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("""
    DROP TABLE IF EXISTS existing_rag_output;
    CREATE TABLE existing_rag_output (
        id SERIAL PRIMARY KEY,
        config_hash TEXT UNIQUE,
        dataset TEXT,
        embedding_model TEXT,
        top_n_retrieval INT,
        semantic_weight FLOAT,
        answer_gen_llm TEXT,
        output JSONB,
        created_at TIMESTAMP DEFAULT NOW()
    );
"""
)

cur.close()
conn.close()

Connected to database: railway


In [2]:
import os
import json
import psycopg2
import warnings
warnings.filterwarnings('ignore')

# check saved output
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("""
    SELECT distinct config_hash from existing_rag_output
""")
rows = cur.fetchall()
print(rows)

cur.execute("""
    SELECT elem
    FROM existing_rag_output,
    jsonb_array_elements(output) AS elem
    LIMIT 1
""")
row = cur.fetchone()
print(json.dumps(row[0], indent=2))

cur.close()
conn.close()

Connected to database: railway
[('c8b119550a0a7fa01a354faef572fec5',), ('bcfe2f64d957b6c94a8cb20b52464cb9',)]
{
  "question": "How to deposit a cheque issued to an associate in my business into my business account?",
  "ai_answer": "To deposit a cheque issued to an associate in your business into your business account, you should first obtain the associate's signature on the back of the cheque. Then, you can deposit it into your business account. However, it's recommended that you have a business account in the name of your business to avoid potential issues. \n\nIf you don't have a business account, you may need to open one, which requires a state-issued DBA certificate and an Employer ID Number (EIN) issued by the IRS. Alternatively, you can try depositing the cheque at the bank where it was drawn, but this may require documentation showing that you are the sole proprietor of the business.",
  "retrieved_lst": [
    {
      "content": "Just have the associate sign the back and then d

### Create Auto Eval Output Storage Table

In [1]:
import os
import psycopg2
import warnings
warnings.filterwarnings('ignore')


DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")

conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

cur.execute("""
    DROP TABLE IF EXISTS existing_auto_eval_output;
    CREATE TABLE existing_auto_eval_output (
        id SERIAL PRIMARY KEY,
        config_hash TEXT UNIQUE,
        retrieval_quality JSONB,
        answer_quality JSONB,
        created_at TIMESTAMP DEFAULT NOW()
    );
"""
)

cur.close()
conn.close()

Connected to database: railway


### Embeddings Generation & Storage

In [18]:
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, VectorStoreIndex
from sqlalchemy import make_url

url = make_url(DATABASE_URL_PUBLIC)

for i in range(len(all_config['embedding_model_settings'])):
    print(f"Processing embedding model: {all_config['embedding_model_settings'][i]['name']}")
    pg_vector_store = PGVectorStore.from_params(
        database=db_name,
        host=url.host,
        password=url.password,
        port=url.port,
        user=url.username,
        table_name=f"embeddings_{all_config['embedding_model_settings'][i]['name']\
                               .split('/')[-1].replace('-', '_').replace('.', 'dot')}",
        embed_dim=all_config['embedding_model_settings'][i]['settings']['embedding_dim'],
    )

    embed_model = HuggingFaceEmbedding(
                    model_name=all_config['embedding_model_settings'][i]['name'], 
                    device="cpu",
                    embed_batch_size=16
                )

    storage_context = StorageContext.from_defaults(vector_store=pg_vector_store)
    index = VectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            show_progress=True,
            embed_model=embed_model
        )

    print("Embeddings saved to Railway Postgres")

Processing embedding model: BAAI/bge-small-en-v1.5


Parsing nodes:   0%|          | 0/30 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Embeddings saved to Railway Postgres


In [10]:
conn = psycopg2.connect(DATABASE_URL_PUBLIC)
cur = conn.cursor()

cur.execute("SELECT current_database();")
print("Connected to database:", cur.fetchone())

cur.execute("""
SELECT table_name FROM information_schema.tables 
WHERE table_schema='public';
""")
print("Tables:", cur.fetchall())

cur.execute("""
SELECT
    relname AS table,
    pg_size_pretty(pg_total_relation_size(relid)) AS size
FROM pg_catalog.pg_statio_user_tables
ORDER BY pg_total_relation_size(relid) DESC;
""")
rows = cur.fetchall()
print("\nTable Sizes:\n")
for row in rows:
    table_name, size = row
    print(f"{table_name:<30} {size}")

cur.close()
conn.close()

Connected to database: ('railway',)
Tables: [('raw_datasets',), ('existing_rag_output',)]

Table Sizes:

raw_datasets                   192 kB
existing_rag_output            24 kB
